In [ ]:
# Model
import os
from pprint import pprint

from langchain_core.callbacks import BaseCallbackHandler
from langchain_core.outputs import LLMResult
from langchain_ollama import ChatOllama

OLLAMA_HOST = os.environ.get("OLLAMA_HOST", "http://localhost:11434")


class TokenCountsCallbackHandler(BaseCallbackHandler):
    def on_llm_end(self, response: LLMResult, **kwargs) -> None:
        for generations in response.generations:
            for generation in generations:
                message = getattr(generation, "message", None)
                if message is not None:
                    meta = getattr(message, "usage_metadata", None)
                    if meta:
                        print(
                            f"Tokens input: {meta.get('input_tokens', 'N/A')}, "
                            f"Tokens output: {meta.get('output_tokens', 'N/A')}, "
                            f"Tokens total: {meta.get('total_tokens', 'N/A')}"
                        )


llm = ChatOllama(
    base_url=OLLAMA_HOST,
    keep_alive="12m",
    model="qwen3.5:0.8b",
    temperature=0.24,
    reasoning=False,
    callbacks=[TokenCountsCallbackHandler()],
)
pprint(llm)

hello_world = llm.invoke("Hello, world!")
hello_world.pretty_print()

In [ ]:
# Search Sources
import os
from pprint import pprint
from typing import NotRequired, TypedDict

import trafilatura
from IPython.display import Image, display
from langchain_community.utilities import SearxSearchWrapper
from langgraph.graph import END, START, StateGraph

from src.prompts import search_sources_prompt_template
from src.subjects import TAROT_CARDS, Subject

SEARXNG_HOST = os.environ.get("SEARXNG_HOST", "http://localhost:8889")


class SubjectState(TypedDict):
    subject: Subject
    query: NotRequired[str]
    sources: NotRequired[str]


def search_sources(state: SubjectState) -> SubjectState:
    print(f"Searching sources for subject: {state.get('subject').name}...")
    subject = state.get("subject")
    query = search_sources_prompt_template.format(
        subject_name=subject.name,
        category_name=subject.category.name,
    )

    searxng_wrapper = SearxSearchWrapper(searx_host=SEARXNG_HOST)
    searxng_results = searxng_wrapper.results(query, num_results=12)

    trafilatura_results = []
    for i, searxng_result in enumerate(searxng_results, start=1):
        url = searxng_result.get("link", "")
        title = searxng_result.get("title", "").strip()
        text = trafilatura.extract(
            trafilatura.fetch_url(url),
            include_comments=False,
            include_tables=True,
            no_fallback=False,
        )
        if not text:
            print(f"Warning: No text extracted from URL: {url}")
            continue
        heading = title if title else url
        trafilatura_results.append(
            f"## Search Result {i}: {heading}\n\n**Source:** [{url}]({url})\n\n{text}"
        )

    if not trafilatura_results:
        raise ValueError(f"No content could be extracted for query: {query}")

    sources = "\n\n---\n\n".join(trafilatura_results)

    return {
        "subject": subject,
        "query": query,
        "sources": sources,
    }


graph_builder = StateGraph(SubjectState)
graph_builder.add_node("search_sources", search_sources)
graph_builder.add_edge(START, "search_sources")
graph_builder.add_edge("search_sources", END)
graph = graph_builder.compile()
display(Image(graph.get_graph().draw_mermaid_png()))

# state = graph.invoke({"subject": TAROT_CARDS[0]})
# pprint(state)

In [ ]:
# Generate Document from Sources
from pprint import pprint
from typing import NotRequired

import inflect
from langchain_core.output_parsers import StrOutputParser

from src.output import strip_markdown_fences, write_document
from src.prompts import affirmations_generate_document_from_sources_prompt_template

inflect_engine = inflect.engine()

str_output_parser = StrOutputParser()


class GenerateDocumentFromSourcesState(SubjectState):
    document: NotRequired[str]


def generate_document(
    state: GenerateDocumentFromSourcesState,
) -> dict[str, str]:
    print(f"Generating document from sources for subject: {state.get('subject').name}...")
    subject = state.get("subject")
    sources = state.get("sources")

    messages = affirmations_generate_document_from_sources_prompt_template.format_messages(
        subject_name=subject.name,
        category_name=subject.category.name,
        category_name_plural=inflect_engine.plural(subject.category.name),  # type: ignore[arg-type]
        sources=sources,
    )

    document = str_output_parser.invoke(llm.invoke(messages))
    document = strip_markdown_fences(document)
    write_document(subject, document, subfolder="affirmations")

    return {"document": document}


graph_builder = StateGraph(GenerateDocumentFromSourcesState)
graph_builder.add_node("search_sources", search_sources)
graph_builder.add_node("generate_document", generate_document)
graph_builder.add_edge(START, "search_sources")
graph_builder.add_edge("search_sources", "generate_document")
graph_builder.add_edge("generate_document", END)
graph = graph_builder.compile()
display(Image(graph.get_graph().draw_mermaid_png()))

# state = graph.invoke({"subject": TAROT_CARDS[0]})
# pprint(state)

In [ ]:
# Generate Document from Source Analysis
from typing import NotRequired

import inflect
from langchain_core.output_parsers import StrOutputParser

from src.prompts import (
    affirmations_analyze_sources_prompt_template,
    affirmations_generate_document_prompt_template,
)

inflect_engine = inflect.engine()

str_output_parser = StrOutputParser()


class GenerateDocumentFromAnalysisState(GenerateDocumentFromSourcesState):
    source_analysis: NotRequired[str]


def analyze_sources(state: GenerateDocumentFromAnalysisState) -> dict[str, str]:
    print(f"Analyzing sources for subject: {state.get('subject').name}")
    subject = state.get("subject")
    messages = affirmations_analyze_sources_prompt_template.format_messages(
        subject_name=subject.name,
        category_name=subject.category.name,
        sources=state.get("sources"),
    )
    source_analysis = str_output_parser.invoke(llm.invoke(messages))
    return {"source_analysis": source_analysis}


def generate_document(state: GenerateDocumentFromAnalysisState) -> dict[str, str]:
    print(f"Generating document from source analysis of subject: {state.get('subject').name}")
    subject = state.get("subject")
    messages = affirmations_generate_document_prompt_template.format_messages(
        subject_name=subject.name,
        category_name=subject.category.name,
        category_name_plural=inflect_engine.plural(subject.category.name),  # type: ignore[arg-type]
        source_analysis=state.get("source_analysis", ""),
    )
    document = str_output_parser.invoke(llm.invoke(messages))
    document = strip_markdown_fences(document)
    write_document(subject, document, subfolder="affirmations")
    return {"document": document}


graph_builder = StateGraph(GenerateDocumentFromAnalysisState)
graph_builder.add_node("search_sources", search_sources)
graph_builder.add_node("analyze_sources", analyze_sources)
graph_builder.add_node("generate_document", generate_document)
graph_builder.add_edge(START, "search_sources")
graph_builder.add_edge("search_sources", "analyze_sources")
graph_builder.add_edge("analyze_sources", "generate_document")
graph_builder.add_edge("generate_document", END)
graph = graph_builder.compile()
display(Image(graph.get_graph().draw_mermaid_png()))

# state = graph.invoke({"subject": TAROT_CARDS[0]})
# pprint(state)

In [ ]:
# Generate Affirmations
import operator
from typing import Annotated, NotRequired, cast

import inflect
import json5 as _json5
from langchain_core.exceptions import OutputParserException
from langchain_core.output_parsers import BaseOutputParser, StrOutputParser
from pydantic import BaseModel as PydanticBaseModel

from src.grammars import GRAMMARS, Grammar
from src.models import (
    Affirmation,
    GeneratedAffirmations,
    GrammarAffirmations,
    SubjectAffirmations,
    ValidationResult,
)
from src.output import write_affirmations_json, write_affirmations_markdown
from src.prompts import (
    affirmations_analyze_document_prompt_template,
    affirmations_generate_affirmations_prompt_template,
    affirmations_validate_affirmation_prompt_template,
)

inflect_engine = inflect.engine()
str_output_parser = StrOutputParser()


class Json5PydanticOutputParser(BaseOutputParser):
    """Output parser using json5 to handle lenient JSON (e.g. trailing commas)."""

    pydantic_object: type[PydanticBaseModel]

    def parse(self, text: str) -> PydanticBaseModel:
        try:
            data = _json5.loads(text)
            return self.pydantic_object.model_validate(data)
        except Exception as e:
            raise OutputParserException(f"Invalid json output: {text}") from e

    @property
    def _type(self) -> str:
        return "json5_pydantic"


class GenerateAffirmationsState(GenerateDocumentFromAnalysisState):
    document_analysis: NotRequired[str]
    grammars: list[Grammar]
    grammar_affirmations: NotRequired[list[GrammarAffirmations]]
    validation_failures: NotRequired[Annotated[list[str], operator.add]]


def analyze_document(state: GenerateAffirmationsState) -> dict[str, str]:
    print(f"Analyzing document for subject: {state.get('subject').name}...")
    messages = affirmations_analyze_document_prompt_template.format_messages(
        subject_name=state.get("subject").name,
        category_name=state.get("subject").category.name,
        document=state.get("document", ""),
    )
    return {"document_analysis": str_output_parser.invoke(llm.invoke(messages))}


def generate_affirmations(
    state: GenerateAffirmationsState,
) -> dict[str, list[GrammarAffirmations]]:
    print(f"Generating affirmations for subject: {state.get('subject').name}...")

    chain = llm.bind(format=GeneratedAffirmations.model_json_schema()) | Json5PydanticOutputParser(
        pydantic_object=GeneratedAffirmations
    )
    grammar_affirmations: list[GrammarAffirmations] = []

    for grammar in state.get("grammars", []):
        print(f"Generating affirmations for grammar: {grammar.name}...")
        messages = affirmations_generate_affirmations_prompt_template.format_messages(
            subject_name=state.get("subject").name,
            document_analysis=state.get("document_analysis", ""),
            grammar_name=grammar.name,
            grammar_description=grammar.description,
            grammar_specifiers=", ".join(grammar.specifiers),
            grammar_examples="; ".join(grammar.examples),
            grammar_emoji=grammar.emoji,
        )
        generated = cast("GeneratedAffirmations", chain.invoke(messages))
        affirmations = [Affirmation(text=text) for text in generated.affirmations]
        grammar_affirmations.append(GrammarAffirmations(grammar=grammar, affirmations=affirmations))

    return {"grammar_affirmations": grammar_affirmations}


def validate_affirmations(
    state: GenerateAffirmationsState,
) -> dict[str, list[str]]:
    print(f"Validating affirmations for subject: {state.get('subject').name}...")

    chain = llm.bind(format=ValidationResult.model_json_schema()) | Json5PydanticOutputParser(
        pydantic_object=ValidationResult
    )
    failures: list[str] = []

    for grammar_affirmations in state.get("grammar_affirmations", []):
        grammar = grammar_affirmations.grammar
        print(f"Validating affirmations for grammar: {grammar.name}...")

        for affirmation in grammar_affirmations.affirmations:
            print(f"Validating affirmation: '{affirmation.text}'...")
            messages = affirmations_validate_affirmation_prompt_template.format_messages(
                affirmation_text=affirmation.text,
                grammar_name=grammar.name,
                grammar_specifiers=", ".join(grammar.specifiers),
                grammar_description=grammar.description,
                grammar_examples="; ".join(grammar.examples),
            )
            try:
                result = cast("ValidationResult", chain.invoke(messages))
            except OutputParserException as e:
                print(f"Parse error validating '{affirmation.text}', skipping: {e}")
                continue

            if not result.valid:
                failure = f"INVALID [{grammar.name}] '{affirmation.text}': {result.reason}"
                failures.append(failure)
                print(f"Validation failure: {failure}")

    if not failures:
        print("Validation: all affirmations passed.")
    else:
        print(f"Validation: {len(failures)} failure(s) logged.")

    return {"validation_failures": failures}


def write_affirmations(state: GenerateAffirmationsState) -> dict:
    subject = state["subject"]
    subject_affirmations = SubjectAffirmations(
        subject=subject,
        grammars=state.get("grammar_affirmations", []),
    )
    write_affirmations_json(subject, subject_affirmations)
    write_affirmations_markdown(subject, subject_affirmations)
    return {}


graph_builder = StateGraph(GenerateAffirmationsState)
graph_builder.add_node("search_sources", search_sources)
graph_builder.add_node("analyze_sources", analyze_sources)
graph_builder.add_node("generate_document", generate_document)
graph_builder.add_node("analyze_document", analyze_document)
graph_builder.add_node("generate_affirmations", generate_affirmations)
graph_builder.add_node("validate_affirmations", validate_affirmations)
graph_builder.add_node("write_affirmations", write_affirmations)
graph_builder.add_edge(START, "search_sources")
graph_builder.add_edge("search_sources", "analyze_sources")
graph_builder.add_edge("analyze_sources", "generate_document")
graph_builder.add_edge("generate_document", "analyze_document")
graph_builder.add_edge("analyze_document", "generate_affirmations")
graph_builder.add_edge("generate_affirmations", "validate_affirmations")
graph_builder.add_edge("validate_affirmations", "write_affirmations")
graph_builder.add_edge("write_affirmations", END)
graph = graph_builder.compile()
display(Image(graph.get_graph().draw_mermaid_png()))


for subject in TAROT_CARDS:
    state = graph.invoke({"subject": subject, "grammars": GRAMMARS})

    print("\n=== SOURCE ANALYSIS ===")
    print(state.get("source_analysis"))

    print("\n=== DOCUMENT ===")
    print(state.get("document", ""))

    print("\n=== DOCUMENT ANALYSIS ===")
    print(state.get("document_analysis"))

    print("\n=== GENERATED AFFIRMATIONS ===")
    for grammar_affirmation in state.get("grammar_affirmations", []):
        print(f"\n{grammar_affirmation.grammar.name}:")
        for affirmation in grammar_affirmation.affirmations:
            print(f"  - {affirmation.text}")

    print("\n=== VALIDATION FAILURES ===")
    for failure in state.get("validation_failures", []):
        print(f"  {failure}")
    if not state.get("validation_failures"):
        print("  None")

## Token Usage & Cost Analysis

Token counts observed from a single run for **The Fool** (tarot card) using `qwen3.5:0.8b` locally.
Costs are estimated for cloud APIs if the same pipeline were run there instead.

In [ ]:
import pandas as pd

# ── Token counts from one observed run (The Fool, qwen3.5:0.8b) ──────────────
# search_sources uses web search only — no LLM tokens
steps = {
    "analyze_sources": {"calls": 1, "input": 4_096, "output": 597},
    "generate_document": {"calls": 1, "input": 903, "output": 1_376},
    "analyze_document": {"calls": 1, "input": 1_518, "output": 207},
    "generate_affirmations": {"calls": 16, "input": 10_106, "output": 871},
    # 48 validation calls total (16 grammars × 3 affirmations).
    # Output was truncated in the log after grammar 8/16 (24 calls observed).
    # Extrapolated: avg 273 input & 129 output per call × 48 calls.
    "validate_affirmations": {"calls": 48, "input": 13_090, "output": 6_206},
}

df_steps = pd.DataFrame(steps).T.astype(int)
df_steps["total"] = df_steps["input"] + df_steps["output"]
df_steps.index.name = "step"

total_input = df_steps["input"].sum()
total_output = df_steps["output"].sum()
total_tokens = df_steps["total"].sum()

print(f"{'Step':<30} {'Calls':>6} {'Input':>8} {'Output':>8} {'Total':>8}")
print("-" * 66)
for step, row in df_steps.iterrows():
    print(
        f"{step:<30} {int(row['calls']):>6} {int(row['input']):>8,} {int(row['output']):>8,} {int(row['total']):>8,}"
    )
print("-" * 66)
print(
    f"{'TOTAL':<30} {int(df_steps['calls'].sum()):>6} {total_input:>8,} {total_output:>8,} {total_tokens:>8,}"
)

In [ ]:
from src.subjects import (
    TAROT_CARDS,
)

# All subject lists — add more as they are created
all_subjects = TAROT_CARDS
total_subjects = len(all_subjects)
print(f"Total subjects: {total_subjects}")

# ── Cloud API pricing  ($ per 1 M tokens, as of April 2026) ───────────────────
# Sources: https://openrouter.ai  (confirmed April 2026)
models = {
    # label                            $/1M input   $/1M output
    # ── OpenAI GPT-5.4 ──────────────────────────────────────
    "GPT-5.4": (2.50, 15.00),
    "GPT-5.4 mini": (0.75, 4.50),
    "GPT-5.4 nano": (0.20, 1.25),
    # ── Google Gemini 2.5 ───────────────────────────────────
    "Gemini 2.5 Pro": (1.25, 10.00),
    "Gemini 2.5 Flash": (0.30, 2.50),
    "Gemini 2.5 Flash-Lite": (0.10, 0.40),
    # ── Google Gemini 3 / 3.1 ───────────────────────────────
    "Gemini 3 Flash Preview": (0.50, 3.00),
    "Gemini 3.1 Flash Lite Preview": (0.25, 1.50),
    "Gemini 3.1 Pro Preview": (2.00, 12.00),
    # ── Anthropic Claude 4.x ────────────────────────────────
    "Claude Haiku 4.5": (1.00, 5.00),
    "Claude Sonnet 4.5 / 4.6": (3.00, 15.00),
    "Claude Opus 4.6": (5.00, 25.00),
    "Claude Opus 4": (15.00, 75.00),
}

rows = []
for name, (price_in, price_out) in models.items():
    cost_per_subject = total_input / 1_000_000 * price_in + total_output / 1_000_000 * price_out
    cost_total = cost_per_subject * total_subjects
    rows.append(
        {
            "Model": name,
            "$/1M input": f"${price_in:.2f}",
            "$/1M output": f"${price_out:.2f}",
            "Cost/subject": f"${cost_per_subject:.4f}",
            f"Cost × {total_subjects} subjects": f"${cost_total:.3f}",
        }
    )

df_costs = pd.DataFrame(rows)
print("\n" + df_costs.to_string(index=False))
print(
    f"\nTokens per subject: {total_input:,} input + {total_output:,} output = {total_tokens:,} total"
)
print("Note: validate_affirmations output is extrapolated from 24/48 observed calls.")